# PSID Data Wrangling Demo

**Companion notebook to Medium Post #2 in the PSID series.**

This notebook turns a raw PSID download into a clean, analysis-ready CSV. You can run it right now — even if you don't have real PSID data yet.

## How it works

The notebook ships with **synthetic data** built into it. When you run the first two cells, the notebook creates a `PSID_Project` folder on your Desktop and writes a fake PSID extract into it. From that point on, every step is identical to what you'd do with real data.

The synthetic extract uses **real PSID variable names** (ER30001, V103, ER82032, and so on) and **real value codes** (1 = Owns, 5 = Rents, etc.). The numbers themselves are fabricated — no real households are represented — but the *structure* is faithful. If you can wrangle this data, you can wrangle the real thing.

## How to run

1. Make sure you have local Jupyter installed (this won't run in Google Colab — it writes to your Desktop).
2. Run the cells in order, top to bottom.
3. When it finishes, open `~/Desktop/PSID_Project/output/psid_clean.csv` to see the result: 430 rows × 25 columns, fully labeled, ready for analysis.

## When you're ready for real data

Edit **Cell 3 only**. Point `EXTRACT_FILE`, `FIMS_FILE`, and `LABELS_FILE` to your real PSID Data Center download. Re-run from Cell 4 onward. The pipeline is identical.

---

*Part of [PSID For the Rest of Us](https://joe-foley.gitbook.io/psid-book) — a practical guide to working with the Panel Study of Income Dynamics in Python.*

In [1]:
import os

# ── Folder Setup ─────────────────────────────────────────────
# ── Folder Setup ─────────────────────────────────────────────
   # Auto-detects the Desktop path on Mac, Linux, and Windows.
# Jupyter on this machine runs as root and cannot auto-detect
# the Desktop, so we hardcode it here.
# If you are on a different machine, update this path.
#DESKTOP = "/Users/joefoley/Desktop"
DESKTOP = os.path.join(os.path.expanduser("~"), "Desktop")
PROJECT  = os.path.join(DESKTOP, "PSID_Project")

RAW_FOLDER    = os.path.join(PROJECT, "raw_data")
OUTPUT_FOLDER = os.path.join(PROJECT, "output")
LABELS_FOLDER = os.path.join(PROJECT, "labels")

for folder in [RAW_FOLDER, OUTPUT_FOLDER, LABELS_FOLDER]:
    os.makedirs(folder, exist_ok=True)
    print(f"  ✓ {folder}")

print()
print("All folders ready.")


  ✓ /home/joe/Desktop/PSID_Project/raw_data
  ✓ /home/joe/Desktop/PSID_Project/output
  ✓ /home/joe/Desktop/PSID_Project/labels

All folders ready.


In [2]:
import pandas as pd
import numpy as np
import random

# ── Reproducibility ──────────────────────────────────────────
np.random.seed(42)
random.seed(42)

N_FAMILIES = 150   # Number of 1968 families
N_PERSONS  = 500   # Total persons across all families

print(f"Generating synthetic PSID extract ({N_PERSONS} persons across {N_FAMILIES} families)...")

# ── Assign persons to families ───────────────────────────────
# Each family gets 2-5 members. Family interview numbers start at 1001.
family_ids = []
person_nums = []
fam_id = 1001

while len(family_ids) < N_PERSONS:
    n_members = random.randint(2, 5)
    n_members = min(n_members, N_PERSONS - len(family_ids))
    for pnum in range(1, n_members + 1):
        family_ids.append(fam_id)
        person_nums.append(pnum)
    fam_id += 1

family_ids = family_ids[:N_PERSONS]
person_nums = person_nums[:N_PERSONS]

# ── Build the synthetic extract DataFrame ────────────────────
df_synth = pd.DataFrame({

    # --- Person identity variables ---
    # ER30001 = 1968 Interview Number (family ID).
    # ER30002 = Person Number within that family.
    # Together they form the unique individual ID: (ER30001 * 1000) + ER30002
    'ER30001': family_ids,                              # 1968 family interview number
    'ER30002': person_nums,                             # Person number within family

    # --- Age in 1968 ---
    'ER30004': np.random.randint(0, 75, N_PERSONS),    # Age in 1968 (0 = infant)

    # --- Sample status ---
    # Codes 1-4 = valid PSID sample members; 5 = non-sample
    'ER32006': np.random.choice([1, 1, 1, 2, 3, 4, 5], N_PERSONS),

    # --- Sex of individual (cross-year summary variable) ---
    # ER32000: 1=Male, 2=Female, 9=NA
    # This is a permanent attribute of each person on the individual file.
    'ER32000': np.random.choice([1, 2], N_PERSONS, p=[0.50, 0.50]),

    # --- Relation to head in 1968 ---
    # ER30003 = Relationship to Head (1968 wave variable)
    # 1=Head, 2=Spouse, 3=Child, 4=Other relative
    'ER30003': np.random.choice([1, 2, 3, 4], N_PERSONS, p=[0.40, 0.30, 0.20, 0.10]),

    # --- Homeownership 1968 (V103) ---
    # 1=Owns or buying, 5=Rents, 8=Neither
    'V103': np.random.choice([1, 5, 8], N_PERSONS, p=[0.65, 0.30, 0.05]),

    # --- Education of head 1968 (V313) ---
    # 1-17 = years of schooling completed
    'V313': np.random.randint(1, 18, N_PERSONS),

    # --- Family income 1967 (V81) ---
    # Reported in whole dollars
    'V81': np.random.lognormal(mean=9.5, sigma=0.8, size=N_PERSONS).astype(int),

    # --- Race of head 1968 (V181) ---
    # 1=White, 2=Black, 3=Other
    'V181': np.random.choice([1, 2, 3], N_PERSONS, p=[0.68, 0.27, 0.05]),

    # --- State 1968 (V93) ---
    # PSID state codes; using a subset of common codes
    'V93': np.random.choice([11, 13, 14, 21, 22, 23, 31, 32, 35, 41, 45], N_PERSONS),

    # --- Sex of 1968 head (V119) ---
    # 1=Male, 2=Female
    'V119': np.random.choice([1, 2], N_PERSONS, p=[0.75, 0.25]),

    # --- Years completed education, 2023 wave (ER35152) ---
    # Highest grade or year of school completed.
    # 1-17 = grade completed; 99 = DK/NA; 0 = Inapplicable
    # Applies to Reference Persons, Spouses/Partners, and OFUMs aged 16+
    'ER35152': np.random.randint(8, 18, N_PERSONS),

    # --- Whether owns or rents home, 2023 wave (ER82032) ---
    # Q: Do you own the home, pay rent, or what?
    # 1=Owns or buying (incl. mobile home owners who rent lots)
    # 5=Pays rent
    # 8=Neither owns nor rents
    'ER82032': np.random.choice([1, 5, 8], N_PERSONS, p=[0.60, 0.35, 0.05]),
})

# ── Save with generic column headers ─────────────────────────
# The real PSID Data Center delivers extracts with generic headers
# (V1, V2, V3 ...). The FIMS file maps these back to real variable names.
real_var_names = list(df_synth.columns)
generic_headers = [f'V{i+1}' for i in range(len(real_var_names))]
df_export = df_synth.copy()
df_export.columns = generic_headers

extract_path = os.path.join(RAW_FOLDER, 'PSID_synthetic_extract.csv')
df_export.to_csv(extract_path, index=False)
print(f"  ✓ Extract saved:  {extract_path}")
print(f"    ({N_PERSONS} rows × {len(real_var_names)} columns, generic headers V1–V{len(real_var_names)})")

# ── Generate the synthetic FIMS file ─────────────────────────
# The FIMS maps each generic column position to its real PSID variable name.
fims_df = pd.DataFrame({
    'name': real_var_names,
    'col_num': list(range(1, len(real_var_names) + 1)),
    'label': [
        '1968 Family Interview Number',       # ER30001
        'Person Number Within Family',        # ER30002
        'Age in 1968',                        # ER30004
        'Sample Status',                      # ER32006
        'Sex of Individual',                  # ER32000
        'Relation to Head 1968',              # ER30003
        'Homeownership Status 1968',          # V103
        'Education of Head 1968',             # V313
        'Family Income 1967',                 # V81
        'Race of Head 1968',                  # V181
        'State 1968',                         # V93
        'Sex of Head 1968',                   # V119
        'Years Completed Education 2023',     # ER35152
        'Whether Owns or Rents Home 2023',    # ER82032
    ]
})

fims_path = os.path.join(RAW_FOLDER, 'PSID_synthetic_FIMS.xlsx')
fims_df.to_excel(fims_path, index=False)
print(f"  ✓ FIMS saved:     {fims_path}")

# ── Generate a minimal labels.txt ────────────────────────────
labels_content = """V103
  1  Owns or buying
  5  Rents
  8  Neither

V181
  1  White
  2  Black
  3  Other

V119
  1  Male
  2  Female

ER32006
  1  Original sample (SRC cross-section)
  2  Born-in sample
  3  Moved-in sample
  4  Joint inclusion sample
  5  Non-sample

ER32000
  1  Male
  2  Female
  9  NA

ER30003
  1  Head
  2  Spouse or Partner
  3  Child
  4  Other relative

ER82032
  1  Owns or buying
  5  Rents
  8  Neither
"""

labels_path = os.path.join(LABELS_FOLDER, 'labels.txt')
with open(labels_path, 'w') as f:
    f.write(labels_content)
print(f"  ✓ Labels saved:   {labels_path}")

print()
print("Synthetic data generation complete. Ready to run the pipeline.")


Generating synthetic PSID extract (500 persons across 150 families)...
  ✓ Extract saved:  /home/joe/Desktop/PSID_Project/raw_data/PSID_synthetic_extract.csv
    (500 rows × 14 columns, generic headers V1–V14)
  ✓ FIMS saved:     /home/joe/Desktop/PSID_Project/raw_data/PSID_synthetic_FIMS.xlsx
  ✓ Labels saved:   /home/joe/Desktop/PSID_Project/labels/labels.txt

Synthetic data generation complete. Ready to run the pipeline.


In [3]:
import os

# ── Configuration ─────────────────────────────────────────────
# This is the ONLY cell you need to update when switching from
# synthetic data to real PSID data.
#
# To use real data:
#   1. Update EXTRACT_FILE to point to your real PSID extract CSV
#   2. Update FIMS_FILE to point to your real FIMS Excel file
#   3. Update LABELS_FILE to point to your real labels.txt
#   4. Re-run all cells from Cell 4 onward.

#DESKTOP = "/Users/joefoley/Desktop"
DESKTOP = os.path.join(os.path.expanduser("~"), "Desktop")
PROJECT  = os.path.join(DESKTOP, "PSID_Project")

RAW_FOLDER    = os.path.join(PROJECT, "raw_data")
OUTPUT_FOLDER = os.path.join(PROJECT, "output")
LABELS_FOLDER = os.path.join(PROJECT, "labels")

EXTRACT_FILE = os.path.join(RAW_FOLDER,    'PSID_synthetic_extract.csv')
FIMS_FILE    = os.path.join(RAW_FOLDER,    'PSID_synthetic_FIMS.xlsx')
LABELS_FILE  = os.path.join(LABELS_FOLDER, 'labels.txt')

print("Configuration:")
print(f"  Extract : {EXTRACT_FILE}")
print(f"  FIMS    : {FIMS_FILE}")
print(f"  Labels  : {LABELS_FILE}")


Configuration:
  Extract : /home/joe/Desktop/PSID_Project/raw_data/PSID_synthetic_extract.csv
  FIMS    : /home/joe/Desktop/PSID_Project/raw_data/PSID_synthetic_FIMS.xlsx
  Labels  : /home/joe/Desktop/PSID_Project/labels/labels.txt


In [4]:
# ── Verify all required files are present ────────────────────
files_to_check = {
    'Extract CSV': EXTRACT_FILE,
    'FIMS Excel':  FIMS_FILE,
    'Labels TXT':  LABELS_FILE,
}

all_ok = True
for label, path in files_to_check.items():
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f"  ✓ {label:15s}  {size_kb:.1f} KB   {path}")
    else:
        print(f"  ✗ MISSING: {label}  →  {path}")
        all_ok = False

print()
if all_ok:
    print("All files present. Proceed to Cell 5.")
else:
    print("Missing files — re-run Cell 2 to regenerate synthetic data.")


  ✓ Extract CSV      18.5 KB   /home/joe/Desktop/PSID_Project/raw_data/PSID_synthetic_extract.csv
  ✓ FIMS Excel       5.2 KB   /home/joe/Desktop/PSID_Project/raw_data/PSID_synthetic_FIMS.xlsx
  ✓ Labels TXT       0.4 KB   /home/joe/Desktop/PSID_Project/labels/labels.txt

All files present. Proceed to Cell 5.


In [5]:
import pandas as pd

# ── Load the FIMS and extract variable name mapping ───────────
# The FIMS Excel file maps generic column numbers to real PSID
# variable names. We auto-detect the column that contains the
# variable names (looks for values starting with 'ER' or 'V').

fims_raw = pd.read_excel(FIMS_FILE)
print("FIMS columns detected:", list(fims_raw.columns))
print(fims_raw.head())

# Auto-detect the variable name column
name_col = None
for col in fims_raw.columns:
    sample = fims_raw[col].dropna().astype(str)
    if sample.str.match(r'^(ER|V)\d+').any():
        name_col = col
        break

if name_col is None:
    raise ValueError("Could not auto-detect variable name column in FIMS file.")

print(f"\nVariable name column identified: '{name_col}'")

# Build mapping: column position → variable name
# col_num is 1-based; generic headers are V1, V2, V3 ...
col_num_col = [c for c in fims_raw.columns if 'col' in c.lower() or 'num' in c.lower()][0]
fims_map = dict(zip(fims_raw[col_num_col], fims_raw[name_col]))
print(f"FIMS mapping loaded: {len(fims_map)} variables")


FIMS columns detected: ['name', 'col_num', 'label']
      name  col_num                         label
0  ER30001        1  1968 Family Interview Number
1  ER30002        2   Person Number Within Family
2  ER30004        3                   Age in 1968
3  ER32006        4                 Sample Status
4  ER32000        5             Sex of Individual

Variable name column identified: 'name'
FIMS mapping loaded: 14 variables


In [6]:
# ── Load extract and rename columns using FIMS ────────────────
df_raw = pd.read_csv(EXTRACT_FILE)
print(f"Extract loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print(f"Generic headers (first 5): {list(df_raw.columns[:5])}")

# Rename V1, V2 ... → real PSID variable names
rename_map = {}
for col in df_raw.columns:
    if col.startswith('V'):
        try:
            num = int(col[1:])
            if num in fims_map:
                rename_map[col] = fims_map[num]
        except ValueError:
            pass

df_raw = df_raw.rename(columns=rename_map)
print(f"\nRenamed {len(rename_map)} columns.")
print(f"Real variable names (first 5): {list(df_raw.columns[:5])}")


Extract loaded: 500 rows × 14 columns
Generic headers (first 5): ['V1', 'V2', 'V3', 'V4', 'V5']

Renamed 14 columns.
Real variable names (first 5): ['ER30001', 'ER30002', 'ER30004', 'ER32006', 'ER32000']


In [7]:
# ── Build unique person IDs ───────────────────────────────────
# The PSID unique individual identifier is:
#   (ER30001 * 1000) + ER30002
# where ER30001 = 1968 Interview Number (family ID)
#       ER30002 = Person Number within family
#
# We also store a zero-padded string version for readability.

df_raw['person_id'] = (df_raw['ER30001'] * 1000 + df_raw['ER30002'])
df_raw['person_id_str'] = (
    df_raw['ER30001'].astype(str).str.zfill(4) + '_' +
    df_raw['ER30002'].astype(str).str.zfill(3)
)

print(f"Person IDs created. Sample:")
print(df_raw[['ER30001', 'ER30002', 'person_id', 'person_id_str']].head(8).to_string(index=False))
print(f"\nUnique persons: {df_raw['person_id'].nunique()}")


Person IDs created. Sample:
 ER30001  ER30002  person_id person_id_str
    1001        1    1001001      1001_001
    1001        2    1001002      1001_002
    1002        1    1002001      1002_001
    1002        2    1002002      1002_002
    1003        1    1003001      1003_001
    1003        2    1003002      1003_002
    1003        3    1003003      1003_003
    1003        4    1003004      1003_004

Unique persons: 500


In [8]:
# ── Filter to valid PSID sample members ──────────────────────
# ER32006 codes:
#   1 = Original sample (SRC cross-section)
#   2 = Born-in sample
#   3 = Moved-in sample
#   4 = Joint inclusion sample
#   5 = Non-sample (exclude)

n_before = len(df_raw)

# Save pre-filter version for reference
pre_filter_path = os.path.join(OUTPUT_FOLDER, 'psid_pre_filter.csv')
df_raw.to_csv(pre_filter_path, index=False)
print(f"Pre-filter CSV saved: {pre_filter_path}  ({n_before} rows)")

# Apply sample filter
df = df_raw[df_raw['ER32006'].isin([1, 2, 3, 4])].copy()
n_after = len(df)

print(f"\nSample filter applied (ER32006 codes 1-4):")
print(f"  Before: {n_before} rows")
print(f"  After:  {n_after} rows  ({n_before - n_after} non-sample removed)")


Pre-filter CSV saved: /home/joe/Desktop/PSID_Project/output/psid_pre_filter.csv  (500 rows)

Sample filter applied (ER32006 codes 1-4):
  Before: 500 rows
  After:  430 rows  (70 non-sample removed)


In [9]:
# ── Build family lineage and role variables ───────────────────
# family_lineage_id: the 1968 family interview number (ER30001)
#   links all descendants back to the founding family unit.
#
# family_role: derived from ER30003 (Relation to Head, 1968)
#   This tells us each person's role in the 1968 family structure.

df['family_lineage_id'] = df['ER30001']

role_map = {1: 'Head', 2: 'Spouse/Partner', 3: 'Child', 4: 'Other relative'}
df['family_role'] = df['ER30003'].map(role_map).fillna('Unknown')

print("Family lineage and role variables created.")
print(df[['ER30001', 'ER30002', 'person_id_str', 'family_lineage_id', 'family_role']].head(10).to_string(index=False))
print(f"\nRole distribution:")
print(df['family_role'].value_counts().to_string())


Family lineage and role variables created.
 ER30001  ER30002 person_id_str  family_lineage_id    family_role
    1001        1      1001_001               1001           Head
    1001        2      1001_002               1001           Head
    1002        1      1002_001               1002 Spouse/Partner
    1002        2      1002_002               1002 Spouse/Partner
    1003        1      1003_001               1003           Head
    1003        2      1003_002               1003 Other relative
    1003        3      1003_003               1003           Head
    1003        4      1003_004               1003 Spouse/Partner
    1004        1      1004_001               1004 Other relative
    1004        2      1004_002               1004 Spouse/Partner

Role distribution:
family_role
Head              168
Spouse/Partner    135
Child              83
Other relative     44


In [10]:
# ── Parse labels.txt and apply to categorical variables ───────
# labels.txt uses the format:
#   VARIABLE_NAME
#     CODE  Label text
#     CODE  Label text

label_dict = {}
current_var = None

with open(LABELS_FILE, 'r') as f:
    for line in f:
        line = line.rstrip()
        if not line:
            continue
        if not line.startswith(' '):
            current_var = line.strip()
            label_dict[current_var] = {}
        else:
            parts = line.strip().split(None, 1)
            if len(parts) == 2:
                try:
                    code = int(parts[0])
                    label_dict[current_var][code] = parts[1]
                except ValueError:
                    pass

print(f"Labels loaded for {len(label_dict)} variables: {list(label_dict.keys())}")

# Apply labels as new _label columns
for var, mapping in label_dict.items():
    if var in df.columns:
        df[f'{var}_label'] = df[var].map(mapping).fillna('Other/Unknown')
        print(f"  ✓ {var}_label created")
    else:
        print(f"  - {var} not in dataframe, skipped")


Labels loaded for 7 variables: ['V103', 'V181', 'V119', 'ER32006', 'ER32000', 'ER30003', 'ER82032']
  ✓ V103_label created
  ✓ V181_label created
  ✓ V119_label created
  ✓ ER32006_label created
  ✓ ER32000_label created
  ✓ ER30003_label created
  ✓ ER82032_label created


In [11]:
# ── Summary of the clean dataset ─────────────────────────────
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print()
print("Columns:")
for col in df.columns:
    n_missing = df[col].isna().sum()
    print(f"  {col:<30s}  dtype={str(df[col].dtype):<10s}  missing={n_missing}")

print()
print("Sample rows:")
display_cols = ['person_id_str', 'family_role', 'ER30004',
                'V103_label', 'V181_label', 'ER32000', 'ER35152', 'ER82032']
available = [c for c in display_cols if c in df.columns]
print(df[available].head(8).to_string(index=False))


Shape: 430 rows × 25 columns

Columns:
  ER30001                         dtype=int64       missing=0
  ER30002                         dtype=int64       missing=0
  ER30004                         dtype=int64       missing=0
  ER32006                         dtype=int64       missing=0
  ER32000                         dtype=int64       missing=0
  ER30003                         dtype=int64       missing=0
  V103                            dtype=int64       missing=0
  V313                            dtype=int64       missing=0
  V81                             dtype=int64       missing=0
  V181                            dtype=int64       missing=0
  V93                             dtype=int64       missing=0
  V119                            dtype=int64       missing=0
  ER35152                         dtype=int64       missing=0
  ER82032                         dtype=int64       missing=0
  person_id                       dtype=int64       missing=0
  person_id_str                

In [12]:
# ── Export the clean, labeled dataset ────────────────────────
clean_path = os.path.join(OUTPUT_FOLDER, 'psid_clean.csv')
df.to_csv(clean_path, index=False)

size_mb = os.path.getsize(clean_path) / (1024 * 1024)
print(f"✓ Clean CSV exported: {clean_path}")
print(f"  {df.shape[0]} rows × {df.shape[1]} columns  ({size_mb:.2f} MB)")
print()
print("─" * 60)
print("TO USE WITH REAL PSID DATA:")
print("─" * 60)
print("1. Open Cell 3 and update the three file paths:")
print("     EXTRACT_FILE  →  your real PSID extract CSV")
print("     FIMS_FILE     →  your real FIMS Excel file")
print("     LABELS_FILE   →  your real labels.txt")
print("2. Delete the PSID_Project folder from your Desktop.")
print("3. Re-run from Cell 4 onward.")
print()
print("NOTE: This notebook must be run in local Jupyter, not Google Colab.")
print("      Colab runs on a remote server and cannot write to your Desktop.")


✓ Clean CSV exported: /home/joe/Desktop/PSID_Project/output/psid_clean.csv
  430 rows × 25 columns  (0.06 MB)

────────────────────────────────────────────────────────────
TO USE WITH REAL PSID DATA:
────────────────────────────────────────────────────────────
1. Open Cell 3 and update the three file paths:
     EXTRACT_FILE  →  your real PSID extract CSV
     FIMS_FILE     →  your real FIMS Excel file
     LABELS_FILE   →  your real labels.txt
2. Delete the PSID_Project folder from your Desktop.
3. Re-run from Cell 4 onward.

NOTE: This notebook must be run in local Jupyter, not Google Colab.
      Colab runs on a remote server and cannot write to your Desktop.
